# 📊 Notebook 1: Data Quality Assessment & Cleaning
**Project:** Savanna Bank Kenya — Customer Churn Analysis  
**Author:** Data Analytics Team  
**Date:** 2026-05-08

---

## Objectives
- Profile the raw dataset and document all data quality issues
- Clean and prepare the data for downstream analysis
- Document every transformation made and why
- Export a clean dataset for use in subsequent notebooks

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.2f}".format)

RAW_PATH   = "../.data/raw/bank_customer_churn_dirty.csv"
CLEAN_PATH = "../.data/cleaned/bank_customer_churn_clean.csv"
VISUALS    = "../outputs/visuals/"

print("Libraries loaded successfully.")

## 2. Load Raw Data

In [ ]:
df_raw = pd.read_csv(RAW_PATH)
print(f"Shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")
df_raw.head()

## 3. Initial Data Profile

Before touching anything, we document the baseline state of the data.

In [ ]:
print("=== COLUMN TYPES ===")
print(df_raw.dtypes)
print()
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)
missing_df = pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})
print("=== MISSING VALUES ===")
print(missing_df[missing_df.missing_count > 0])

In [ ]:
df_raw.describe().T

In [ ]:
cat_cols = ["gender", "region", "branch", "marital_status",
            "employment_status", "account_type", "loan_type"]
for col in cat_cols:
    print(f"--- {col} ({df_raw[col].nunique()} unique) ---")
    print(df_raw[col].value_counts(dropna=False).head(10))
    print()

## 4. Data Quality Issues Found

| # | Column | Issue | Fix Applied |
|---|--------|-------|-------------|
| 1 | `gender` | Inconsistent encoding (e.g. "M", "male", "Male") | Standardised to "Male" / "Female" |
| 2 | `region` | Leading/trailing whitespace | `.strip()` applied |
| 3 | `credit_score` | Sentinel value 999 used for missing | Replaced with column median |
| 4 | `credit_score` | Null values | Imputed with median |
| 5 | `account_balance_ksh` | Negative balances (invalid) | Replaced with 0 |
| 6 | `monthly_income_ksh` | Extreme outliers (>99th percentile) | Capped at 99th percentile |
| 7 | `last_transaction_date` | Future dates beyond today | Replaced with median date |
| 8 | `marital_status` | Missing values | Imputed with mode |
| 9 | `employment_status` | Missing values | Imputed with mode |
| 10 | Duplicates | Duplicate `customer_id` rows | Dropped, keeping first occurrence |

## 5. Cleaning Steps

In [ ]:
df = df_raw.copy()

# Issue 10: Remove duplicate rows
before = len(df)
df = df.drop_duplicates(subset="customer_id", keep="first")
print(f"Duplicates removed: {before - len(df)}")

In [ ]:
# Issue 2: Strip whitespace from region
df["region"] = df["region"].str.strip()
print("Regions after cleaning:", sorted(df["region"].unique()))

In [ ]:
# Issue 1: Standardise gender encoding
df["gender"] = df["gender"].str.strip().str.title()
df["gender"] = df["gender"].replace({"M": "Male", "F": "Female"})
print("Gender values:", df["gender"].unique())

In [ ]:
# Issues 3 & 4: Fix credit_score sentinel 999 and nulls
df["credit_score"] = df["credit_score"].replace(999, np.nan)
median_cs = df["credit_score"].median()
df["credit_score"] = df["credit_score"].fillna(median_cs)
print(f"Credit score nulls after fix: {df['credit_score'].isnull().sum()}")
print(f"Max credit score: {df['credit_score'].max()}")

In [ ]:
# Issue 5: Replace negative balances with 0
neg_count = (df["account_balance_ksh"] < 0).sum()
df["account_balance_ksh"] = df["account_balance_ksh"].clip(lower=0)
print(f"Negative balances replaced: {neg_count}")

In [ ]:
# Issue 6: Cap monthly_income_ksh outliers at 99th percentile
p99 = df["monthly_income_ksh"].quantile(0.99)
outlier_count = (df["monthly_income_ksh"] > p99).sum()
df["monthly_income_ksh"] = df["monthly_income_ksh"].clip(upper=p99)
print(f"Income outliers capped: {outlier_count} (cap at KSH {p99:,.0f})")

In [ ]:
# Issue 7: Fix future last_transaction_date
df["last_transaction_date"] = pd.to_datetime(df["last_transaction_date"], errors="coerce")
df["account_open_date"]     = pd.to_datetime(df["account_open_date"], errors="coerce")
today = pd.Timestamp("2026-05-08")
future_mask = df["last_transaction_date"] > today
median_date = df.loc[~future_mask, "last_transaction_date"].median()
df.loc[future_mask, "last_transaction_date"] = median_date
print(f"Future dates corrected: {future_mask.sum()}")

In [ ]:
# Issues 8 & 9: Impute missing categorical values with mode
for col in ["marital_status", "employment_status"]:
    mode_val = df[col].mode()[0]
    n_missing = df[col].isnull().sum()
    df[col] = df[col].fillna(mode_val)
    print(f"{col}: {n_missing} nulls filled with '{mode_val}'")

## 6. Feature Engineering

Deriving meaningful predictive features as specified in the project brief.

In [ ]:
# days_since_last_txn
df["days_since_last_txn"] = (today - df["last_transaction_date"]).dt.days

# account_age_months
df["account_age_months"] = (
    (today.year  - df["account_open_date"].dt.year) * 12 +
    (today.month - df["account_open_date"].dt.month)
).astype(float)

# has_loan flag
df["has_loan"] = (df["loan_type"] != "No Loan").astype(int)

# loan_to_income_ratio
df["loan_to_income_ratio"] = np.where(
    df["has_loan"] == 1,
    df["loan_amount_ksh"] / df["monthly_income_ksh"].replace(0, np.nan),
    0
).astype(float)

# repayment_burden: monthly repayment / monthly income
df["repayment_burden"] = (
    df["monthly_repayment_ksh"] / df["monthly_income_ksh"].replace(0, np.nan)
).fillna(0)

# balance_to_income_ratio
df["balance_to_income_ratio"] = (
    df["account_balance_ksh"] / df["monthly_income_ksh"].replace(0, np.nan)
).fillna(0)

print("Engineered features added successfully.")
print(df[["days_since_last_txn","account_age_months","has_loan",
          "loan_to_income_ratio","repayment_burden","balance_to_income_ratio"]].head())

## 7. Cleaning Summary Visualisation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

missing_before = df_raw.isnull().sum()
missing_after  = df.isnull().sum()
compare = pd.DataFrame({"Before": missing_before, "After": missing_after})
compare = compare[compare["Before"] > 0]
compare.plot(kind="bar", ax=axes[0], color=["#E24B4A", "#1D9E75"], edgecolor="white")
axes[0].set_title("Missing Values: Before vs After Cleaning", fontweight="bold")
axes[0].set_ylabel("Count")
axes[0].tick_params(axis="x", rotation=45)
axes[0].legend()

counts = pd.Series({"Raw rows": len(df_raw), "Clean rows": len(df)})
counts.plot(kind="bar", ax=axes[1], color=["#378ADD", "#1D9E75"], edgecolor="white")
axes[1].set_title("Row Count: Before vs After Deduplication", fontweight="bold")
axes[1].set_ylabel("Count")
axes[1].tick_params(axis="x", rotation=0)
for idx, v in enumerate(counts):
    axes[1].text(idx, v + 5, f"{v:,}", ha="center", fontweight="bold")

plt.tight_layout()
plt.savefig(VISUALS + "02_missing_values_before_after.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved.")

## 8. Export Clean Dataset

In [ ]:
df.to_csv(CLEAN_PATH, index=False)
print(f"Clean dataset saved: {len(df):,} rows x {df.shape[1]} columns")
print(f"Path: {CLEAN_PATH}")
print()
print("Final columns:", list(df.columns))